In [2]:
import os
from pyspark.sql import SparkSession, functions as F

# Имя каталога
catalog = "lk"

# Доступ к minio
access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
warehouse = os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab")

#Настройка каталога в Spark
spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config(f"spark.sql.legacy.parquet.nanosAsLong","true") #Добавили для поддержки Timestamp из паркета
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)

#Уровень логирования
spark.sparkContext.setLogLevel("WARN")

In [8]:
# Создаем namespace DDS
spark.sql("CREATE NAMESPACE dds")

DataFrame[]

In [9]:
# Создаем таблицу с требуемой структурой
spark.sql("""CREATE TABLE dds.users (
  `id` BIGINT NOT NULL,
  `first_name` STRING,
  `last_name` STRING,
  `phone` STRING,
  `sex` STRING,
  `birth_date` DATE,
  `start_date` DATE NOT NULL,
  `end_date` DATE NOT NULL,
  `last_updated` DATE NOT NULL
)
USING iceberg""")

DataFrame[]

In [3]:
spark.sql("""truncate table dds.users""")
# Загружаем в таблицу расчитанные данные 
## для заполнения по умолчанию используем COALESCE
## для вычисления периода используем оконную функцию LEAD
spark.sql("""INSERT INTO dds.users
  SELECT `id`,
  `first_name`,
  `last_name`,
  `phone`,
  `sex`,
  `birth_date`,
  case (rank() over (PARTITION BY id ORDER BY last_updated)) 
     when 1 then `birth_date` 
     else `last_updated` 
  end AS `start_date`,
  COALESCE(LEAD(last_updated) OVER (PARTITION BY id ORDER BY last_updated),DATE('2500-01-01')) AS `end_date`,
    current_timestamp() AS `last_updated`
  FROM stage.users
    """)

DataFrame[]